In [26]:
"""
Callset benchmarking and comparison.
"""

import copy
import importlib
import re
import pickle
import warnings
from pathlib import Path
from collections import defaultdict
import time
import gzip
import shutil

import numpy as np
import pandas as pd
import matplotlib as mpl
import pyranges as pr
import pysam
import seaborn as sns
import shap
from scipy.spatial import cKDTree
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import os
from typing import Union, Tuple, Optional, List, Any, Literal, Dict


workdir = "/vast/ai_projects/Papenfuss_SVEnsemble2"
os.chdir(workdir)
os.getcwd()

import svensemble.vcfparse as vcfp # noqa: E402 (flake8)
import svensemble.breakends as bnds
import svensemble.breakpoints as bps
import svensemble.events as evts
import svensemble.caller_specific as caller_specific
import svensemble.benchmark as benchmark
import svensemble._caller_params as caller_params
import svensemble._sample_params as sample_params
import svensemble.chunk_match as chunk_match
import svensemble.matched_breakpoints as matchedbps
import svensemble.utils as sv_utils
import svensemble.importances as importances

importlib.reload(vcfp)
importlib.reload(bnds)
importlib.reload(bps)
importlib.reload(evts)
importlib.reload(caller_specific)
importlib.reload(benchmark)
importlib.reload(caller_params)
importlib.reload(sample_params)
importlib.reload(chunk_match)
importlib.reload(matchedbps)
importlib.reload(sv_utils)

pd.set_option('display.max_columns', None)


In [30]:
sample_name_orig = 'NA12878'
sample_name_new = 'NA12878_test'
callsets = ['gridss', 'manta_candidate', 'manta_diploid', 'delly']

n_vars = 100

write_modes = {
    '.vcf': 'w',
    '.gz': 'w',
    '.bcf': 'wb'
}

sample_config_orig = sv_utils.create_config_dict(sample_name_orig)
sample_config_new = sv_utils.create_config_dict(sample_name_new)

for callset_name in callsets:
    callset_vcf_path_orig = sample_config_orig[sample_name_orig]['callsets'][callset_name]['path']
    vcf_file_orig = pysam.VariantFile(callset_vcf_path_orig)

    # Get total number of variants in the callset
    total_variants = sum(1 for _ in vcf_file_orig)
    vcf_file_orig.reset()
    print(f"Total number of variants in {callset_name}: {total_variants}")

    # Get 100 random indices for the callset
    random_indices = np.random.choice(total_variants, size=n_vars, replace=False)
    # ensure unique, sorted, 0-based integer indices
    indices_iter = iter(sorted(int(x) for x in random_indices))
    next_idx = next(indices_iter, None)


    # Iterate over the vcf file again, pick
    callset_vcf_path_new = sample_config_new[sample_name_new]['callsets'][callset_name]['path']
    # Check if the new callset vcf path directory exists, if not create it
    callset_vcf_path_new_dir = os.path.dirname(callset_vcf_path_new)
    if not os.path.exists(callset_vcf_path_new_dir):
        os.makedirs(callset_vcf_path_new_dir)

    # Infer write mode from the file extension
    write_mode = write_modes[os.path.splitext(callset_vcf_path_new)[1]]

    vcf_file_new = pysam.VariantFile(callset_vcf_path_new, write_mode, header=vcf_file_orig.header)
    for i, rec in enumerate(vcf_file_orig):
        if next_idx is None:
            break
        if i == next_idx:
            vcf_file_new.write(rec)
            next_idx = next(indices_iter, None)

    # If the original was gzipped, compress the new file
    if callset_vcf_path_orig.suffix == '.gz':
        # First, rename the file to temp
        temp_path = Path(str(callset_vcf_path_new) + '.temp')
        os.rename(callset_vcf_path_new, temp_path)
        # Then, compress the temp file
        with open(temp_path, 'rb') as f_in:
            with gzip.open(Path(str(callset_vcf_path_new) + '.gz'), 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        # Then, delete the temp file
        os.remove(temp_path)

    vcf_file_new.close()

Adding custom columns to the config dict for sample NA12878
Custom columns: []
Adding custom columns to the config dict for sample NA12878_test
Custom columns: []
Total number of variants in gridss: 863933
Total number of variants in manta_candidate: 118066
Total number of variants in manta_diploid: 8351
Total number of variants in delly: 32078
